In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

UPLOAD_DIR = Path(".")

aws_cold = pd.read_csv(UPLOAD_DIR / "aws_cold_aggregated.csv")
gcp_cold = pd.read_csv(UPLOAD_DIR / "gcp_cold_aggregated.csv")
aws_warm = pd.read_csv(UPLOAD_DIR / "aws_warm_aggregated.csv")
gcp_warm = pd.read_csv(UPLOAD_DIR / "gcp_warm_aggregated.csv")

# Functions present in both cold data and fig1 basis function set
TASKS = [
    ("a_passthrough",     "P"),
    ("b_compress",        "C"),
    ("c_decompress",      "D"),
    ("e_matrix_multiply", "M"),
]
MEMORIES = [128, 256, 512, 768, 1024, 1536, 2048]
PAYLOAD   = "medium"

def get_residual(cold_df, warm_df, task, mem, payload):
    c = cold_df[(cold_df['task']==task) & (cold_df['payload_size']==payload) &
                (cold_df['memory_mb']==mem)]['invocation_latency_ms'].values
    w = warm_df[(warm_df['task']==task) & (warm_df['payload_size']==payload) &
                (warm_df['memory_mb']==mem)]['invocation_latency_ms'].values
    if len(c) == 0 or len(w) == 0:
        return None, None
    residuals = c - np.median(w)
    return np.median(residuals) / 1000, np.std(residuals) / 1000

print("Setup complete.")
print(f"AWS cold: {len(aws_cold)} rows | AWS warm: {len(aws_warm)} rows")
print(f"GCP cold: {len(gcp_cold)} rows | GCP warm: {len(gcp_warm)} rows")


Setup complete.
AWS cold: 670 rows | AWS warm: 1046 rows
GCP cold: 775 rows | GCP warm: 995 rows


## Data preview

In [19]:
for provider, cold_df, warm_df in [("AWS", aws_cold, aws_warm), ("GCP", gcp_cold, gcp_warm)]:
    print(f"{provider} -- median(cold - warm) [s] at micro payload:")
    print(f"{'ID':<4} " + " ".join(f"{m:>7}" for m in MEMORIES))
    print("-" * 58)
    for task, letter in TASKS:
        row = []
        for mem in MEMORIES:
            m_val, _ = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
            row.append(f"{m_val:7.3f}" if m_val is not None else "    ---")
        print(f"{letter:<4} {''.join(row)}")
    print()


AWS -- median(cold - warm) [s] at micro payload:
ID       128     256     512     768    1024    1536    2048
----------------------------------------------------------
P      7.820  3.734  2.181  1.416  1.238  1.048  0.932
C        ---  4.120  2.294  1.519  1.267  1.031  0.938
D        ---  4.681  2.411  1.752  1.456  1.000  1.140
M        ---    ---  2.570  2.090  1.535  1.305  1.500

GCP -- median(cold - warm) [s] at micro payload:
ID       128     256     512     768    1024    1536    2048
----------------------------------------------------------
P        ----11.062 -3.755 -0.804 -0.992  0.478 -0.093
C        ---    --- -9.120 -2.710 -2.186  0.560  0.597
D        ---    --- -4.583 -0.997 -1.116  0.191 -1.158
M        ---    --- -3.891 -1.865 -3.607  0.511 -0.141



## AWS -- P

In [24]:
cold_df, warm_df = aws_cold, aws_warm
task = "a_passthrough"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% AWS P cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% AWS P cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (128, 7.8197)
  (256, 3.7338)
  (512, 2.1815)
  (768, 1.4161)
  (1024, 1.2385)
  (1536, 1.0481)
  (2048, 0.9320)
};


## AWS -- C

In [25]:
cold_df, warm_df = aws_cold, aws_warm
task = "b_compress"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% AWS C cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=triangle*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% AWS C cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (256, 4.1205)
  (512, 2.2943)
  (768, 1.5187)
  (1024, 1.2675)
  (1536, 1.0313)
  (2048, 0.9381)
};


## AWS -- D

In [32]:
cold_df, warm_df = aws_cold, aws_warm
task = "c_decompress"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% AWS D cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=square*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% AWS D cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=square*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (256, 4.6813)
  (512, 2.4113)
  (768, 1.7521)
  (1024, 1.4561)
  (1536, 1.0001)
  (2048, 1.1402)
};


## AWS -- M

In [27]:
cold_df, warm_df = aws_cold, aws_warm
task = "e_matrix_multiply"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% AWS M cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=diamond*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% AWS M cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (512, 2.5697)
  (768, 2.0901)
  (1024, 1.5345)
  (1536, 1.3055)
  (2048, 1.5002)
};


## GCP -- P

In [28]:
cold_df, warm_df = gcp_cold, gcp_warm
task = "a_passthrough"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% GCP P cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% GCP P cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (256, -11.0622)
  (512, -3.7546)
  (768, -0.8045)
  (1024, -0.9922)
  (1536, 0.4777)
  (2048, -0.0930)
};


## GCP -- C

In [29]:
cold_df, warm_df = gcp_cold, gcp_warm
task = "b_compress"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% GCP C cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=triangle*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% GCP C cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (512, -9.1204)
  (768, -2.7096)
  (1024, -2.1862)
  (1536, 0.5598)
  (2048, 0.5974)
};


## GCP -- D

In [30]:
cold_df, warm_df = gcp_cold, gcp_warm
task = "c_decompress"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% GCP D cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=square*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% GCP D cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (512, -4.5828)
  (768, -0.9973)
  (1024, -1.1162)
  (1536, 0.1907)
  (2048, -1.1581)
};


## GCP -- M

In [31]:
cold_df, warm_df = gcp_cold, gcp_warm
task = "e_matrix_multiply"

coords, errs = [], []
for mem in MEMORIES:
    m_val, s_val = get_residual(cold_df, warm_df, task, mem, PAYLOAD)
    if m_val is not None:
        coords.append((mem, m_val))
        errs.append(s_val)

lines = [
    f"% GCP M cold-warm residual ({PAYLOAD} payload, mean +/- 1 std)",
    r"\addplot+[",
    r"  mark=diamond*, error bars/.cd, y dir=both, y explicit,",
    r"] coordinates {",
]
for (mem, val), err in zip(coords, errs):
    lines.append(f"  ({mem}, {val:.4f})")
lines.append(r"};")
print("\n".join(lines))

% GCP M cold-warm residual (medium payload, mean +/- 1 std)
\addplot+[
  mark=*, error bars/.cd, y dir=both, y explicit,
] coordinates {
  (512, -3.8908)
  (768, -1.8654)
  (1024, -3.6068)
  (1536, 0.5111)
  (2048, -0.1411)
};
